In [1]:
# ============================================================
# CELL 1 — PROJECT SETUP AND REQUIRED LIBRARIES
# ============================================================

from pathlib import Path
import sys

import pandas as pd

# The notebook is located inside the notebooks/ folder.
# Therefore, the project root is one level above it.
PROJECT_ROOT = Path.cwd().parent

# Add the project root to Python's import path.
# This allows us to import our reusable functions from src/.
sys.path.append(str(PROJECT_ROOT))

# Import the general SOLETE loader.
#
# The loader is resolution-independent:
# the input HDF5 file determines whether we are loading
# 60-min, 5-min, 1-min, or 1-sec data.
from src.data_loader import load_solete

In [2]:
# ============================================================
# CELL 2 — LOAD THE 60-MINUTE SOLETE DATASET
# ============================================================

# Path to the 60-minute SOLETE HDF5 file.
dataset_path = Path("../solete_dataset/SOLETE_Pombo_60min.h5")

# Load the dataset using our reusable loader.
df = load_solete(dataset_path)

# Display the DataFrame so we can visually inspect
# the loaded data.
df

,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-11-17 00:00:00+00:00,13.953556,1.6000,7.257889,315.208889,0.0,0.0,0.0,0.0,3000.0,0.0,0.0
2018-11-17 01:00:00+00:00,13.130778,1.6000,7.106611,311.173333,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 02:00:00+00:00,12.132889,1.6000,7.641944,310.871111,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 03:00:00+00:00,11.424944,1.6000,6.390722,299.809444,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 04:00:00+00:00,10.999722,1.6000,6.414944,307.498333,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2018-09-12 19:00:00+00:00,13.417528,0.7415,3.435722,265.097778,0.0,0.0,0.0,0.0,1000.0,0.0,0.0
2018-09-12 20:00:00+00:00,12.902444,0.8000,3.179056,258.788889,0.0,0.0,0.0,0.0,1000.0,0.0,0.0
2018-09-12 21:00:00+00:00,12.327500,0.8000,2.792444,248.180278,0.0,0.0,0.0,0.0,1000.0,0.0,0.0


In [3]:
# ============================================================
# CELL 3 — BASIC DATASET STRUCTURE
# ============================================================

# Check the number of rows and columns in the dataset.
#
# The result is:
#     (number_of_rows, number_of_columns)
#
# Each row represents one hourly observation.
# Each column represents one measured variable.
df.shape

(10969, 11)

In [4]:
# ============================================================
# CELL 4 — IDENTIFY THE DATASET VARIABLES
# ============================================================

# Display all column names in the dataset.
#
# Each column represents one environmental, solar, wind,
# astronomical, or power-related measurement.
#
# Understanding these variables is important because later
# we need to decide:
#   1. Which variables will be input features?
#   2. Which variable will be our prediction target?
#   3. Which variables may need cleaning or transformation?
df.columns.tolist()

['TEMPERATURE[degC]',
 'HUMIDITY[%]',
 'WIND_SPEED[m1s]',
 'WIND_DIR[deg]',
 'GHI[kW1m2]',
 'POA Irr[kW1m2]',
 'P_Gaia[kW]',
 'P_Solar[kW]',
 'Pressure[mbar]',
 'Azimuth[deg]',
 'Elevation[deg]']

In [5]:
# ============================================================
# CELL 5 — UNDERSTAND WHAT EACH VARIABLE REPRESENTS
# ============================================================

# The SOLETE dataset contains environmental measurements,
# solar measurements, wind measurements, and power output.
#
# We create a reference table so that the meaning of every
# variable is documented directly inside our notebook.

variable_description = {
    "TEMPERATURE[degC]": "Air temperature in degrees Celsius",
    "HUMIDITY[%]": "Relative humidity",
    "WIND_SPEED[m1s]": "Wind speed",
    "WIND_DIR[deg]": "Wind direction in degrees",
    "GHI[kW1m2]": "Global Horizontal Irradiance",
    "POA Irr[kW1m2]": "Plane-of-Array irradiance",
    "P_Gaia[kW]": "Electrical power generated by the Gaia wind turbine",
    "P_Solar[kW]": "Electrical power generated by the solar PV system",
    "Pressure[mbar]": "Atmospheric pressure",
    "Azimuth[deg]": "Solar azimuth angle",
    "Elevation[deg]": "Solar elevation angle",
}


# Convert the dictionary into a DataFrame with two columns:
#   Variable    -> original SOLETE variable name
#   Description -> explanation of that variable
variable_table = pd.DataFrame(
    list(variable_description.items()),
    columns=["Variable", "Description"]
)


# Display the reference table.
variable_table

,Variable,Description
0,TEMPERATURE[degC],Air temperature in degrees Celsius
1,HUMIDITY[%],Relative humidity
2,WIND_SPEED[m1s],Wind speed
3,WIND_DIR[deg],Wind direction in degrees
4,GHI[kW1m2],Global Horizontal Irradiance
5,POA Irr[kW1m2],Plane-of-Array irradiance
6,P_Gaia[kW],Electrical power generated by the Gaia wind tu...
7,P_Solar[kW],Electrical power generated by the solar PV system
8,Pressure[mbar],Atmospheric pressure
9,Azimuth[deg],Solar azimuth angle


In [6]:
# ============================================================
# CELL 6 — INSPECT THE TIMESTAMP INDEX
# ============================================================

# The SOLETE dataset uses Timestamp as its index.
#
# Since this is a time-series forecasting project, understanding
# the timestamps is extremely important.
#
# Later, we will derive features such as:
#   - Year
#   - Month
#   - Day
#   - Hour
#   - Day of week
#   - Solar/astronomical information
#   - Vedic calendar information
#
# Before doing any feature engineering, we first need to see
# what the timestamps actually look like.

# Display the first 10 timestamps.
df.index[:10]

DatetimeIndex(['2018-11-17 00:00:00+00:00', '2018-11-17 01:00:00+00:00',
               '2018-11-17 02:00:00+00:00', '2018-11-17 03:00:00+00:00',
               '2018-11-17 04:00:00+00:00', '2018-11-17 05:00:00+00:00',
               '2018-11-17 06:00:00+00:00', '2018-11-17 07:00:00+00:00',
               '2018-11-17 08:00:00+00:00', '2018-11-17 09:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Timestamp', freq=None)

In [7]:
# ============================================================
# CELL 7 — CHECK THE TIME PERIOD COVERED BY THE DATASET
# ============================================================

# Find the earliest timestamp in the dataset.
start_time = df.index.min()

# Find the latest timestamp in the dataset.
end_time = df.index.max()


# Display the complete observation period.
#
# This tells us exactly which dates are available for our
# forecasting experiments and Vedic-calendar conversion.
print("Start timestamp :", start_time)
print("End timestamp   :", end_time)

Start timestamp : 2018-06-01 00:00:00+00:00
End timestamp   : 2019-09-01 00:00:00+00:00


In [8]:
# ============================================================
# CELL 8 — CHECK CHRONOLOGICAL ORDER OF TIMESTAMPS
# ============================================================

# A forecasting dataset should normally be ordered from the
# earliest observation to the latest observation.
#
# We check this before doing any time-series analysis because
# incorrectly ordered timestamps can cause problems when:
#   - creating lag features
#   - splitting training and testing data
#   - calculating time differences
#   - performing forecasting experiments


# Check whether every timestamp is greater than or equal to
# the timestamp immediately before it.
is_sorted = df.index.is_monotonic_increasing


# Display the result.
print("Are timestamps in chronological order?", is_sorted)

Are timestamps in chronological order? False


In [9]:
# ============================================================
# CELL 9 — FIND WHERE TIMESTAMP ORDERING BREAKS
# ============================================================

# Calculate the time difference between every timestamp and
# the timestamp immediately before it.
#
# For correctly ordered hourly data, we would normally expect:
#
#     1 hour
#
# A negative difference means that the timestamp has moved
# backwards in time.
time_difference = df.index.to_series().diff()


# Find all locations where the timestamp moves backwards.
# These are the points we need to investigate before deciding
# whether the dataset should be sorted.
backward_jumps = time_difference[time_difference < pd.Timedelta(0)]


# Display the first 20 backward jumps.
backward_jumps.head(20)

Timestamp
2018-09-17 00:00:00+00:00   -107 days +00:00:00
2018-08-14 00:00:00+00:00    -41 days +00:00:00
2019-04-01 00:00:00+00:00    -88 days +00:00:00
2018-08-30 00:00:00+00:00   -215 days +00:00:00
2018-09-10 00:00:00+00:00   -238 days +00:00:00
2019-04-14 00:00:00+00:00    -84 days +00:00:00
2019-01-16 00:00:00+00:00   -164 days +00:00:00
2019-05-07 00:00:00+00:00    -48 days +00:00:00
2019-03-22 00:00:00+00:00   -140 days +00:00:00
2019-04-15 01:00:00+00:00     -5 days +01:00:00
2018-08-16 00:00:00+00:00   -243 days +00:00:00
2018-07-21 00:00:00+00:00    -27 days +00:00:00
2019-07-03 00:00:00+00:00    -38 days +00:00:00
2018-10-08 00:00:00+00:00   -269 days +00:00:00
2019-04-13 00:00:00+00:00   -130 days +00:00:00
2018-12-28 00:00:00+00:00   -107 days +01:00:00
2018-07-15 00:00:00+00:00   -167 days +00:00:00
2019-06-22 00:00:00+00:00    -10 days +00:00:00
2019-01-06 00:00:00+00:00   -197 days +00:00:00
2018-12-08 00:00:00+00:00    -30 days +00:00:00
Name: Timestamp, dtype: timede

In [10]:
# ============================================================
# CELL 10 — COUNT TIMESTAMP ORDERING BREAKS
# ============================================================

# We already calculated the time difference between consecutive
# timestamps in Cell 9.
#
# A negative time difference means the dataset moves backwards
# in time at that point.
#
# We count how many such backward jumps exist.

number_of_backward_jumps = (
    time_difference < pd.Timedelta(0)
).sum()


# Display the total number of ordering breaks.
print("Number of backward timestamp jumps:", number_of_backward_jumps)


# Also display the total number of observations.
print("Total number of observations:", len(df))

Number of backward timestamp jumps: 239
Total number of observations: 10969


In [11]:
# ============================================================
# CELL 11 — INSPECT TIME GAP DISTRIBUTION
# ============================================================

# The variable 'time_difference' contains the time difference
# between each timestamp and the previous timestamp.
#
# For a normal hourly sequence, the expected difference is:
#
#     1 hour
#
# Large positive gaps indicate that the dataset jumps forward
# between separate observation periods.
#
# Negative gaps indicate that the next observation belongs to
# an earlier time period, which we already observed in Cell 9.


# Count how frequently each time difference occurs.
gap_counts = time_difference.value_counts()


# Display the 20 most common time differences.
gap_counts.head(20)

Timestamp
0 days 01:00:00        10512
-165 days +01:00:00        4
27 days 00:00:00           3
-144 days +01:00:00        3
-10 days +01:00:00         3
-161 days +01:00:00        3
-48 days +01:00:00         3
42 days 02:00:00           3
-112 days +02:00:00        3
5 days 00:00:00            2
-48 days +00:00:00         2
-5 days +01:00:00          2
-38 days +00:00:00         2
-130 days +00:00:00        2
-2 days +00:00:00          2
110 days 01:00:00          2
-133 days +00:00:00        2
-154 days +01:00:00        2
188 days 00:00:00          2
-150 days +00:00:00        2
Name: count, dtype: int64

In [12]:
# ============================================================
# CELL 12 — IDENTIFY TIME-SERIES BLOCKS
# ============================================================

# A new block begins whenever the timestamp does not continue
# normally from the previous observation.
#
# For this hourly dataset, a normal continuation is:
#
#     previous timestamp + 1 hour
#
# Therefore, any difference other than exactly 1 hour marks
# a possible boundary between separate observation blocks.


# Identify positions where the time difference is not 1 hour.
block_boundaries = time_difference != pd.Timedelta(hours=1)


# Count the number of detected block boundaries.
number_of_boundaries = block_boundaries.sum()


# Display the result.
print("Number of detected block boundaries:", number_of_boundaries)


# Each boundary starts a new block.
# We add these boundary indicators cumulatively to create
# a block number for every observation.
block_id = block_boundaries.cumsum()


# Display how many observations belong to each block.
block_sizes = block_id.value_counts().sort_index()

print("\nNumber of observations in each block:")
print(block_sizes)

Number of detected block boundaries: 457

Number of observations in each block:
Timestamp
1      25
2      25
3      25
4      25
5      25
       ..
453    23
454    23
455    23
456    23
457    23
Name: count, Length: 457, dtype: int64


In [13]:
# ============================================================
# CELL 13 — INSPECT THE FIRST FEW TIME-SERIES BLOCKS
# ============================================================

# 'block_id' identifies which observations belong to the same
# continuous block.
#
# We now inspect the first 10 blocks.
#
# For each block, we will display:
#   - block number
#   - number of observations
#   - first timestamp
#   - last timestamp
#
# This helps us understand whether the blocks represent
# approximately one-day measurement periods.

block_summary = (
    df.index.to_series()
    .groupby(block_id)
    .agg(
        observations="size",
        start_time="min",
        end_time="max"
    )
)


# Add the block number as a normal column so that the table
# is easier to read.
block_summary.insert(
    0,
    "block_id",
    block_summary.index
)


# Display the first 10 blocks.
block_summary.head(10)

,block_id,observations,start_time,end_time
Timestamp,,,,
1,1,25,2018-11-17 00:00:00+00:00,2018-11-18 00:00:00+00:00
2,2,25,2019-01-01 00:00:00+00:00,2019-01-02 00:00:00+00:00
3,3,25,2018-09-17 00:00:00+00:00,2018-09-18 00:00:00+00:00
4,4,25,2018-09-23 00:00:00+00:00,2018-09-24 00:00:00+00:00
5,5,25,2018-08-14 00:00:00+00:00,2018-08-15 00:00:00+00:00
6,6,25,2019-06-27 00:00:00+00:00,2019-06-28 00:00:00+00:00
7,7,25,2019-04-01 00:00:00+00:00,2019-04-02 00:00:00+00:00
8,8,25,2018-08-30 00:00:00+00:00,2018-08-31 00:00:00+00:00
9,9,25,2019-05-05 00:00:00+00:00,2019-05-06 00:00:00+00:00


In [14]:
# ============================================================
# CELL 14 — CHECK CONTINUITY INSIDE EACH BLOCK
# ============================================================

# We know that most blocks contain 25 observations covering
# approximately 24 hours.
#
# Now we check whether the observations INSIDE each block are
# actually separated by exactly one hour.
#
# This is important because a block could contain 25 observations
# but still have a missing or duplicated hourly timestamp.


# Calculate the time difference between consecutive observations
# within each block.
within_block_difference = (
    df.index.to_series()
    .groupby(block_id)
    .diff()
)


# Count all differences that are NOT exactly one hour.
irregular_within_block = within_block_difference[
    within_block_difference.notna()
    & (within_block_difference != pd.Timedelta(hours=1))
]


# Display the number of irregular intervals.
print(
    "Irregular intervals inside blocks:",
    len(irregular_within_block)
)


# Display a few examples if any exist.
print("\nExamples:")
print(irregular_within_block.head(20))

Irregular intervals inside blocks: 0

Examples:
Series([], Name: Timestamp, dtype: timedelta64[ns])


In [15]:
# ============================================================
# CELL 15 — CHECK THE DURATION OF EACH DETECTED BLOCK
# ============================================================

# Each detected block should contain a continuous sequence of
# hourly observations.
#
# We now calculate the duration of every block:
#
#     duration = last timestamp - first timestamp
#
# A block containing 25 hourly observations should normally
# have a duration of exactly 24 hours.


block_summary["duration"] = (
    block_summary["end_time"]
    - block_summary["start_time"]
)


# Count how many blocks have each duration.
duration_counts = block_summary["duration"].value_counts()


# Display the most common durations.
duration_counts.head(15)

duration
0 days 23:00:00    156
1 days 00:00:00    151
0 days 22:00:00    150
Name: count, dtype: int64

In [16]:
# ============================================================
# CELL 16 — INSPECT BLOCK SIZES AND DATES
# ============================================================

# We now create a compact table showing:
#   - block number
#   - number of observations
#   - start date
#   - end date
#   - duration
#
# This allows us to see whether the shorter blocks are related
# to particular dates or periods.

block_inspection = block_summary[
    [
        "block_id",
        "observations",
        "start_time",
        "end_time",
        "duration"
    ]
].copy()


# Display the first 20 detected blocks.
block_inspection.head(20)

,block_id,observations,start_time,end_time,duration
Timestamp,,,,,
1,1,25,2018-11-17 00:00:00+00:00,2018-11-18 00:00:00+00:00,1 days 00:00:00
2,2,25,2019-01-01 00:00:00+00:00,2019-01-02 00:00:00+00:00,1 days 00:00:00
3,3,25,2018-09-17 00:00:00+00:00,2018-09-18 00:00:00+00:00,1 days 00:00:00
4,4,25,2018-09-23 00:00:00+00:00,2018-09-24 00:00:00+00:00,1 days 00:00:00
5,5,25,2018-08-14 00:00:00+00:00,2018-08-15 00:00:00+00:00,1 days 00:00:00
6,6,25,2019-06-27 00:00:00+00:00,2019-06-28 00:00:00+00:00,1 days 00:00:00
7,7,25,2019-04-01 00:00:00+00:00,2019-04-02 00:00:00+00:00,1 days 00:00:00
8,8,25,2018-08-30 00:00:00+00:00,2018-08-31 00:00:00+00:00,1 days 00:00:00
9,9,25,2019-05-05 00:00:00+00:00,2019-05-06 00:00:00+00:00,1 days 00:00:00


In [17]:
# ============================================================
# CELL 17 — CHECK FOR DUPLICATE TIMESTAMPS
# ============================================================

# A timestamp should normally identify one unique observation.
#
# Duplicate timestamps can cause problems later when we:
#   - sort the dataset
#   - create lag features
#   - resample the time series
#   - train forecasting models
#
# We therefore check whether the SOLETE dataset contains
# any repeated timestamps.

duplicate_timestamps = df.index.duplicated()


# Count the number of duplicated observations.
number_of_duplicates = duplicate_timestamps.sum()


# Display the result.
print("Number of duplicate timestamps:", number_of_duplicates)

Number of duplicate timestamps: 0


In [18]:
# ============================================================
# CELL 18 — CREATE A CHRONOLOGICALLY SORTED COPY
# ============================================================

# The original DataFrame 'df' preserves the order in which
# observations were stored in the SOLETE HDF5 file.
#
# We do NOT modify 'df'.
#
# For forecasting, however, we need observations arranged
# chronologically so that:
#     earlier observations -> later observations
#
# We therefore create a separate sorted DataFrame.


# Sort the observations according to their Timestamp index.
df_sorted = df.sort_index()


# Display the first 10 timestamps after sorting.
print("First 10 timestamps after sorting:")
print(df_sorted.index[:10])


# Display the last 10 timestamps after sorting.
print("\nLast 10 timestamps after sorting:")
print(df_sorted.index[-10:])


# Confirm that the sorted DataFrame is now chronological.
print(
    "\nAre timestamps chronological?",
    df_sorted.index.is_monotonic_increasing
)

First 10 timestamps after sorting:
DatetimeIndex(['2018-06-01 00:00:00+00:00', '2018-06-01 01:00:00+00:00',
               '2018-06-01 02:00:00+00:00', '2018-06-01 03:00:00+00:00',
               '2018-06-01 04:00:00+00:00', '2018-06-01 05:00:00+00:00',
               '2018-06-01 06:00:00+00:00', '2018-06-01 07:00:00+00:00',
               '2018-06-01 08:00:00+00:00', '2018-06-01 09:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Timestamp', freq=None)

Last 10 timestamps after sorting:
DatetimeIndex(['2019-08-31 15:00:00+00:00', '2019-08-31 16:00:00+00:00',
               '2019-08-31 17:00:00+00:00', '2019-08-31 18:00:00+00:00',
               '2019-08-31 19:00:00+00:00', '2019-08-31 20:00:00+00:00',
               '2019-08-31 21:00:00+00:00', '2019-08-31 22:00:00+00:00',
               '2019-08-31 23:00:00+00:00', '2019-09-01 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Timestamp', freq=None)

Are timestamps chronological? True


In [19]:
# ============================================================
# CELL 19 — CHECK TIME GAPS AFTER SORTING
# ============================================================

# Now that the dataset is in chronological order, we can
# calculate the time difference between consecutive observations.
#
# For a perfectly continuous hourly dataset, every difference
# would be exactly:
#
#     1 hour
#
# If we find larger gaps, they indicate periods where no
# observation is present.


# Calculate the difference between consecutive timestamps.
sorted_time_difference = df_sorted.index.to_series().diff()


# Count how often each time difference occurs.
sorted_gap_counts = sorted_time_difference.value_counts()


# Display the most common time differences.
sorted_gap_counts.head(20)

Timestamp
0 days 01:00:00    10968
Name: count, dtype: int64

In [20]:
# ============================================================
# CELL 20 — FINAL TIMESTAMP VALIDATION
# ============================================================

# At this point, we have:
#   - removed no data
#   - preserved the original DataFrame 'df'
#   - created the chronological copy 'df_sorted'
#
# We now perform a final validation of the sorted time series.


# ------------------------------------------------------------
# Check 1 — Chronological order
# ------------------------------------------------------------

print(
    "Chronological order:",
    df_sorted.index.is_monotonic_increasing
)


# ------------------------------------------------------------
# Check 2 — Duplicate timestamps
# ------------------------------------------------------------

print(
    "Duplicate timestamps:",
    df_sorted.index.duplicated().sum()
)


# ------------------------------------------------------------
# Check 3 — Number of observations
# ------------------------------------------------------------

print(
    "Number of observations:",
    len(df_sorted)
)


# ------------------------------------------------------------
# Check 4 — Time intervals
# ------------------------------------------------------------

# Calculate all consecutive time differences.
final_time_difference = df_sorted.index.to_series().diff()


# Ignore the first NaN value and check whether every remaining
# interval is exactly one hour.
all_hourly = (
    final_time_difference.iloc[1:]
    == pd.Timedelta(hours=1)
).all()


print(
    "All consecutive intervals are 1 hour:",
    all_hourly
)

Chronological order: True
Duplicate timestamps: 0
Number of observations: 10969
All consecutive intervals are 1 hour: True


In [21]:
# ============================================================
# CELL 21 — BASIC STATISTICS OF THE CHRONOLOGICALLY SORTED DATA
# ============================================================

# We now move from timestamp validation to understanding the
# actual measurements in the SOLETE dataset.
#
# The describe() function gives us useful statistics such as:
#
#   count  -> number of observations
#   mean   -> average value
#   std    -> standard deviation
#   min    -> minimum value
#   25%    -> first quartile
#   50%    -> median
#   75%    -> third quartile
#   max    -> maximum value
#
# We use df_sorted because this is the chronological version
# that we will eventually use for time-series analysis.


summary_statistics = df_sorted.describe().T


# Display the statistics.
#
# Transposing with .T makes each variable appear as one row,
# which makes the table easier to inspect.
summary_statistics

,count,mean,std,min,25%,50%,75%,max
TEMPERATURE[degC],10969.0,12.075792,7.347850,-4.402528,5.737250,12.595083,17.651250,44.241984
HUMIDITY[%],10969.0,0.782769,0.195906,0.000000,0.686722,0.800000,0.900000,2.700000
WIND_SPEED[m1s],10969.0,3.642386,2.141412,0.000000,2.117694,3.319972,4.699583,23.150403
WIND_DIR[deg],10969.0,212.288659,75.736164,0.000000,146.496111,217.555000,276.340278,639.344818
GHI[kW1m2],10969.0,0.140451,0.212350,0.000000,0.000000,0.012351,0.220647,0.910141
POA Irr[kW1m2],10969.0,0.144488,0.237810,0.000000,0.000000,0.010066,0.186332,0.961514
P_Gaia[kW],10969.0,0.016259,0.323335,0.000000,0.000000,0.000000,0.000000,10.184867
P_Solar[kW],10969.0,1.029960,1.657731,0.000000,0.000000,0.077733,1.400071,6.975178
Pressure[mbar],10969.0,1043.845751,205.662219,992.367136,1000.000000,1000.000000,1000.000000,3000.000000
Azimuth[deg],10969.0,-0.003834,0.994408,-56.907645,0.000000,0.000000,0.000000,44.794950


In [22]:
# ============================================================
# CELL 22 — CHECK POTENTIALLY UNUSUAL VALUE RANGES
# ============================================================

# We now check the measurements against simple physical ranges.
#
# IMPORTANT:
# These are NOT automatic cleaning rules.
#
# We are only using them to identify observations that deserve
# further investigation.
#
# We will NOT modify or delete any observations in this cell.


# ------------------------------------------------------------
# Temperature
# ------------------------------------------------------------
# Temperatures below -50°C or above 60°C would be unusual
# for this location and measurement context.
temperature_unusual = (
    (df_sorted["TEMPERATURE[degC]"] < -50)
    | (df_sorted["TEMPERATURE[degC]"] > 60)
)


# ------------------------------------------------------------
# Humidity
# ------------------------------------------------------------
# Relative humidity is normally expected between 0% and 100%.
humidity_unusual = (
    (df_sorted["HUMIDITY[%]"] < 0)
    | (df_sorted["HUMIDITY[%]"] > 100)
)


# ------------------------------------------------------------
# Wind speed
# ------------------------------------------------------------
# Wind speed cannot physically be negative.
wind_speed_unusual = (
    df_sorted["WIND_SPEED[m1s]"] < 0
)


# ------------------------------------------------------------
# Wind direction
# ------------------------------------------------------------
# A conventional wind-direction representation is normally
# between 0° and 360°.
wind_direction_unusual = (
    (df_sorted["WIND_DIR[deg]"] < 0)
    | (df_sorted["WIND_DIR[deg]"] > 360)
)


# ------------------------------------------------------------
# Irradiance
# ------------------------------------------------------------
# Irradiance cannot physically be negative.
ghi_unusual = (
    df_sorted["GHI[kW1m2]"] < 0
)

poa_unusual = (
    df_sorted["POA Irr[kW1m2]"] < 0
)


# ------------------------------------------------------------
# Power
# ------------------------------------------------------------
# Electrical power cannot normally be negative for these
# generated-power measurements.
gaia_power_unusual = (
    df_sorted["P_Gaia[kW]"] < 0
)

solar_power_unusual = (
    df_sorted["P_Solar[kW]"] < 0
)


# ------------------------------------------------------------
# Pressure
# ------------------------------------------------------------
# We use a broad physical screening range here.
# This is deliberately NOT a final validity rule.
pressure_unusual = (
    (df_sorted["Pressure[mbar]"] < 800)
    | (df_sorted["Pressure[mbar]"] > 1100)
)


# ------------------------------------------------------------
# Solar angles
# ------------------------------------------------------------
# Solar elevation is normally between -90° and +90°.
elevation_unusual = (
    (df_sorted["Elevation[deg]"] < -90)
    | (df_sorted["Elevation[deg]"] > 90)
)


# ------------------------------------------------------------
# Display the number of flagged observations
# ------------------------------------------------------------

print("Potentially unusual observations:")
print("----------------------------------")

print(
    "Temperature      :",
    temperature_unusual.sum()
)

print(
    "Humidity         :",
    humidity_unusual.sum()
)

print(
    "Wind speed       :",
    wind_speed_unusual.sum()
)

print(
    "Wind direction   :",
    wind_direction_unusual.sum()
)

print(
    "GHI              :",
    ghi_unusual.sum()
)

print(
    "POA irradiance   :",
    poa_unusual.sum()
)

print(
    "Gaia power       :",
    gaia_power_unusual.sum()
)

print(
    "Solar power      :",
    solar_power_unusual.sum()
)

print(
    "Pressure         :",
    pressure_unusual.sum()
)

print(
    "Solar elevation  :",
    elevation_unusual.sum()
)

Potentially unusual observations:
----------------------------------
Temperature      : 0
Humidity         : 0
Wind speed       : 0
Wind direction   : 103
GHI              : 0
POA irradiance   : 0
Gaia power       : 0
Solar power      : 0
Pressure         : 479
Solar elevation  : 0


In [23]:
# ============================================================
# CELL 23 — INSPECT FLAGGED WIND DIRECTION AND PRESSURE VALUES
# ============================================================

# Instead of changing the suspicious observations, we first
# look at the actual values and their timestamps.
#
# This helps us determine whether these are:
#   - isolated unusual measurements,
#   - repeated sensor values,
#   - a scaling/encoding issue,
#   - or something that is actually valid for this dataset.

print("WIND DIRECTION VALUES ABOVE 360°")
print("--------------------------------")
print(
    df_sorted.loc[
        df_sorted["WIND_DIR[deg]"] > 360,
        ["WIND_DIR[deg]"]
    ].head(20)
)

print("\nPRESSURE VALUES ABOVE 1100 mbar")
print("--------------------------------")
print(
    df_sorted.loc[
        df_sorted["Pressure[mbar]"] > 1100,
        ["Pressure[mbar]"]
    ].head(20)
)

WIND DIRECTION VALUES ABOVE 360°
--------------------------------
                           WIND_DIR[deg]
Timestamp                               
2018-06-04 00:00:00+00:00     536.147819
2018-06-12 00:00:00+00:00     572.041400
2018-06-13 00:00:00+00:00     497.312309
2018-06-14 00:00:00+00:00     489.970825
2018-06-18 00:00:00+00:00     456.465685
2018-06-21 00:00:00+00:00     446.032231
2018-06-22 00:00:00+00:00     537.536816
2018-06-24 00:00:00+00:00     458.721867
2018-06-29 00:00:00+00:00     542.864962
2018-07-10 00:00:00+00:00     488.997499
2018-07-14 00:00:00+00:00     615.971103
2018-07-19 00:00:00+00:00     615.106141
2018-07-20 00:00:00+00:00     629.986663
2018-07-21 00:00:00+00:00     514.706863
2018-07-26 00:00:00+00:00     544.922201
2018-08-02 00:00:00+00:00     583.533482
2018-08-05 00:00:00+00:00     586.353154
2018-08-06 00:00:00+00:00     603.231453
2018-08-15 00:00:00+00:00     608.551542
2018-08-16 00:00:00+00:00     392.021395

PRESSURE VALUES ABOVE 1100 mbar

In [24]:
# ============================================================
# CELL 24 — INVESTIGATE REPEATED WIND-DIRECTION AND PRESSURE VALUES
# ============================================================

# We saw that some wind-direction values exceed 360° and that
# pressure contains many repeated values such as exactly 2000.
#
# Before deciding whether these are errors, we investigate
# their frequency and persistence.

print("WIND DIRECTION VALUES ABOVE 360°")
print("--------------------------------")

wind_direction_flagged = df_sorted.loc[
    df_sorted["WIND_DIR[deg]"] > 360,
    "WIND_DIR[deg]"
]

print("Number of observations:", len(wind_direction_flagged))
print("Number of unique values:", wind_direction_flagged.nunique())

print("\nFirst 20 values:")
print(wind_direction_flagged.head(20))


print("\n\nPRESSURE VALUE FREQUENCIES")
print("--------------------------")

pressure_counts = (
    df_sorted["Pressure[mbar]"]
    .value_counts()
    .sort_index()
)

print(pressure_counts)


print("\n\nMOST COMMON PRESSURE VALUES")
print("---------------------------")

print(
    df_sorted["Pressure[mbar]"]
    .value_counts()
    .head(15)
)

WIND DIRECTION VALUES ABOVE 360°
--------------------------------
Number of observations: 103
Number of unique values: 103

First 20 values:
Timestamp
2018-06-04 00:00:00+00:00    536.147819
2018-06-12 00:00:00+00:00    572.041400
2018-06-13 00:00:00+00:00    497.312309
2018-06-14 00:00:00+00:00    489.970825
2018-06-18 00:00:00+00:00    456.465685
2018-06-21 00:00:00+00:00    446.032231
2018-06-22 00:00:00+00:00    537.536816
2018-06-24 00:00:00+00:00    458.721867
2018-06-29 00:00:00+00:00    542.864962
2018-07-10 00:00:00+00:00    488.997499
2018-07-14 00:00:00+00:00    615.971103
2018-07-19 00:00:00+00:00    615.106141
2018-07-20 00:00:00+00:00    629.986663
2018-07-21 00:00:00+00:00    514.706863
2018-07-26 00:00:00+00:00    544.922201
2018-08-02 00:00:00+00:00    583.533482
2018-08-05 00:00:00+00:00    586.353154
2018-08-06 00:00:00+00:00    603.231453
2018-08-15 00:00:00+00:00    608.551542
2018-08-16 00:00:00+00:00    392.021395
Name: WIND_DIR[deg], dtype: float64


PRESSURE VA

In [25]:
# ============================================================
# CELL 25 — CHECK WHEN THE SUSPICIOUS PRESSURE VALUES OCCUR
# ============================================================

# We already know that 2000 and 3000 mbar occur many times.
#
# Now we check their timestamps.
#
# If these values appear in long continuous periods, that would
# suggest a systematic sensor/data issue rather than random noise.
#
# We still do NOT modify the data.

suspicious_pressure = df_sorted[
    df_sorted["Pressure[mbar]"] > 1100
][["Pressure[mbar]"]]

print("Number of suspicious pressure observations:")
print(len(suspicious_pressure))

print("\nFirst 30 suspicious observations:")
print(suspicious_pressure.head(30))

print("\nLast 30 suspicious observations:")
print(suspicious_pressure.tail(30))

Number of suspicious pressure observations:
479

First 30 suspicious observations:
                           Pressure[mbar]
Timestamp                                
2018-06-02 00:00:00+00:00          2000.0
2018-06-03 00:00:00+00:00          2000.0
2018-06-04 00:00:00+00:00          2000.0
2018-06-05 00:00:00+00:00          2000.0
2018-06-06 00:00:00+00:00          2000.0
2018-06-07 00:00:00+00:00          2000.0
2018-06-08 00:00:00+00:00          2000.0
2018-06-09 00:00:00+00:00          2000.0
2018-06-10 00:00:00+00:00          2000.0
2018-06-11 00:00:00+00:00          2000.0
2018-06-12 00:00:00+00:00          2000.0
2018-06-13 00:00:00+00:00          2000.0
2018-06-14 00:00:00+00:00          2000.0
2018-06-15 00:00:00+00:00          2000.0
2018-06-16 00:00:00+00:00          2000.0
2018-06-17 00:00:00+00:00          2000.0
2018-06-18 00:00:00+00:00          2000.0
2018-06-19 00:00:00+00:00          2000.0
2018-06-20 00:00:00+00:00          2000.0
2018-06-21 00:00:00+00:00          

In [26]:
# ============================================================
# CELL 26 — COMPARE NORMAL AND SUSPICIOUS PRESSURE PERIODS
# ============================================================

# We now compare the other weather and power measurements when
# pressure is normal versus when pressure is suspicious.
#
# This is NOT a cleaning step.
#
# The purpose is to see whether the 2000/3000 pressure values
# occur while the rest of the sensors continue producing
# realistic measurements.

normal_pressure = df_sorted["Pressure[mbar]"] <= 1100
suspicious_pressure = df_sorted["Pressure[mbar]"] > 1100

comparison_columns = [
    "TEMPERATURE[degC]",
    "HUMIDITY[%]",
    "WIND_SPEED[m1s]",
    "WIND_DIR[deg]",
    "GHI[kW1m2]",
    "POA Irr[kW1m2]",
    "P_Gaia[kW]",
    "P_Solar[kW]",
]

comparison = pd.DataFrame({
    "Normal pressure": df_sorted.loc[
        normal_pressure, comparison_columns
    ].mean(),

    "Suspicious pressure": df_sorted.loc[
        suspicious_pressure, comparison_columns
    ].mean(),
})

comparison

,Normal pressure,Suspicious pressure
TEMPERATURE[degC],12.024410,13.201039
HUMIDITY[%],0.764371,1.185670
WIND_SPEED[m1s],3.611430,4.320313
WIND_DIR[deg],209.618215,270.770824
GHI[kW1m2],0.146732,0.002884
POA Irr[kW1m2],0.150870,0.004729
P_Gaia[kW],0.016372,0.013789
P_Solar[kW],1.075391,0.035029


In [27]:
# ============================================================
# CELL 27 — CHECK WIND-DIRECTION VALUES AS CYCLIC ANGLES
# ============================================================

# Wind direction is an angular variable.
#
# A direction of 0° and 360° represent the same direction.
# Therefore, one useful diagnostic is to calculate the
# equivalent direction inside the standard 0°–360° range.
#
# IMPORTANT:
# We are NOT replacing the original values.
# We are only creating a temporary diagnostic column.

wind_direction_check = df_sorted.loc[
    df_sorted["WIND_DIR[deg]"] > 360,
    ["WIND_DIR[deg]"]
].copy()

wind_direction_check["Equivalent_0_360"] = (
    wind_direction_check["WIND_DIR[deg]"] % 360
)

wind_direction_check.head(20)

,WIND_DIR[deg],Equivalent_0_360
Timestamp,,
2018-06-04 00:00:00+00:00,536.147819,176.147819
2018-06-12 00:00:00+00:00,572.041400,212.041400
2018-06-13 00:00:00+00:00,497.312309,137.312309
2018-06-14 00:00:00+00:00,489.970825,129.970825
2018-06-18 00:00:00+00:00,456.465685,96.465685
2018-06-21 00:00:00+00:00,446.032231,86.032231
2018-06-22 00:00:00+00:00,537.536816,177.536816
2018-06-24 00:00:00+00:00,458.721867,98.721867
2018-06-29 00:00:00+00:00,542.864962,182.864962


In [28]:
# ============================================================
# CELL 28 — CHECK THE COMPLETE WIND-DIRECTION RANGE
# ============================================================

# We now apply modulo 360 to ALL wind-direction observations.
#
# This is only a diagnostic copy.
# The original WIND_DIR[deg] column remains unchanged.
#
# After modulo 360, every direction should fall within:
#       0° <= direction < 360°

wind_direction_mod360 = (
    df_sorted["WIND_DIR[deg]"] % 360
)

print("Original wind-direction range:")
print(
    df_sorted["WIND_DIR[deg]"].min(),
    "to",
    df_sorted["WIND_DIR[deg]"].max()
)

print("\nModulo-360 wind-direction range:")
print(
    wind_direction_mod360.min(),
    "to",
    wind_direction_mod360.max()
)

print("\nNumber of values outside [0°, 360°):")
print(
    (
        (wind_direction_mod360 < 0)
        | (wind_direction_mod360 >= 360)
    ).sum()
)

Original wind-direction range:
0.0 to 639.3448180050013

Modulo-360 wind-direction range:
0.0 to 358.7821617115866

Number of values outside [0°, 360°):
0


## 60-Minute EDA — Main Findings

The exploratory analysis of the 60-minute SOLETE dataset produced the
following main observations.

### Dataset structure

- 10,969 observations and 11 measured variables.
- Data cover June 2018 to September 2019.
- No missing values were found.
- No duplicate timestamps were found.
- The timestamps were not originally stored in complete chronological
  order.
- After sorting the timestamp index, all consecutive observations were
  separated by exactly one hour.
- Therefore, chronological analyses should use the sorted DataFrame
  rather than relying on the original row order.

### Solar and wind variables

- The dataset contains both renewable-energy targets:
  - `P_Solar[kW]`
  - `P_Gaia[kW]`
- Solar irradiance variables (`GHI` and `POA Irr`) contain no negative
  values.
- Solar power and wind power also contain no negative values.
- Some distributions, particularly `P_Gaia`, `Azimuth`, and
  `Elevation`, differ noticeably from those observed later in the
  higher-resolution dataset.
- These differences will be examined explicitly during the
  cross-resolution comparison rather than being corrected at this
  stage.

### Wind-direction behaviour

- 103 wind-direction observations were greater than 360 degrees.
- The maximum raw value was approximately 639.34 degrees.
- Applying modulo 360 to a diagnostic copy converted all observations
  into the conventional range:

      0 <= wind direction < 360

- The original values were not overwritten.
- For forecasting, wind direction should later be represented as a
  circular variable, for example using sine and cosine transformations.

### Pressure behaviour

- Pressure contained a substantial number of unusually high values.
- Most normal-looking observations were near 1000 mbar.
- Suspicious values included values near:
  - 2000 mbar
  - 3000 mbar
- These values occurred systematically rather than as isolated random
  spikes.
- The raw pressure data will therefore remain unchanged during EDA.
- Pressure cleaning, exclusion, or alternative preprocessing should be
  evaluated later as part of the forecasting pipeline.

### Data-handling decision

The purpose of this notebook is dataset understanding rather than data
correction.

No suspicious source values were permanently modified.

Potential preprocessing decisions will be documented and tested during
the modelling stage instead of silently altering the original SOLETE
dataset.

### Next step

The higher-resolution SOLETE datasets will be examined using the same
essential structural checks.

After completing the resolution-specific EDA notebooks, a dedicated
cross-resolution comparison will be used to determine:

- which characteristics are common across resolutions,
- which apparent anomalies are resolution-specific,
- how the target distributions change with sampling resolution,
- and which temporal resolution is most appropriate for the main
  forecasting experiments.